In [1]:
import subprocess, sys

libs = [
    ["torch==2.1.2", "torchvision==0.16.2", "--index-url", "https://download.pytorch.org/whl/cu118"],
    ["diffusers==0.31.0"],
    ["transformers==4.40.0"],
    ["accelerate==0.30.0"],
    ["numpy<2.0"],
    ["Pillow"]
]

for lib in libs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + lib)

In [2]:
import torch

device = "cuda"
dtype  = torch.float16

torch.backends.cudnn.benchmark = True

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"dtype: {dtype}")

GPU  : NVIDIA GeForce RTX 3060
VRAM : 12.5 GB
dtype: torch.float16


In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type="dpmsolver++",
    use_karras_sigmas=True
)

pipe = pipe.to(device)
pipe.enable_attention_slicing(1)

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xformers: on")
except Exception:
    print("xformers: off")

print("Model ready — GPU")

Loading pipeline components...: 100%|██████████| 5/5 [00:02<00:00,  1.94it/s]


xformers: off
Model ready — GPU


In [7]:
import time

prompt = (
    "A sprawling cyberpunk megacity at midnight, rain-slicked streets reflecting "
    "cascades of neon signs in Cyrillic and Japanese, towering brutalist skyscrapers "
    "wrapped in holographic banners, hovercars threading between lit windows, "
    "volumetric fog, ultra-detailed, cinematic 4k, photorealistic render"
)

negative_prompt = (
    "daytime, sunny, cartoon, anime, low quality, blurry, watermark, "
    "text overlay, deformed architecture, oversaturated"
)

generator = torch.Generator(device=device).manual_seed(42)

t0 = time.time()

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=25,
    guidance_scale=7.5,
    width=512,
    height=512,
    generator=generator
).images[0]

elapsed = time.time() - t0

image.save("m_gpu.png")
print(f"Done in {elapsed:.1f}s  |  saved: m_gpu.png")
image

  0%|          | 0/25 [00:00<?, ?it/s]


TypeError: argument of type 'NoneType' is not iterable